# Dodatek 10.1: Řetězení promptů

- [Lekce](#lesson)
- [Příkladové hřiště](#example-playground)

## Nastavení

Spusťte následující buňku nastavení pro načtení vašeho API klíče a vytvoření pomocné funkce `get_completion`.

In [ ]:
%pip install anthropic --quiet

# Import vestavěnou knihovnu regulárních výrazů v Pythonu
import re
from anthropic import AnthropicBedrock

%store -r MODEL_NAME
%store -r AWS_REGION

client = AnthropicBedrock(aws_region=AWS_REGION)

def get_completion(messages, system_prompt=''):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=messages,
        system=system_prompt
    )
    return message.content[0].text

---

## Lekce

Říká se, že "psaní je přepisování." Ukazuje se, že **Claude může často zlepšit přesnost své odpovědi, když je o to požádán**!

Existuje mnoho způsobů, jak Claude požádat, aby "přemýšlel znovu". Způsoby, které se zdají přirozené, když žádáme člověka, aby si svou práci zkontroloval, budou obecně fungovat i pro Claude. (Podívejte se na naši [dokumentaci k prompt chaining](https://docs.anthropic.com/claude/docs/chain-prompts) pro další příklady, kdy a jak používat prompt chaining.)

### Příklady

V tomto příkladu požádáme Claude, aby přišel s deseti slovy... ale jedno nebo více z nich není skutečné slovo.

In [ ]:
```python
# Počáteční prompt
first_user = "Name ten words that all end with the exact letters 'ab'."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Uložení a vytištění odpovědi od Claude
first_response = get_completion(messages)
print(first_response)
```

**Požádání Claude, aby jeho odpověď byla přesnější** opravuje chybu!

Níže jsme stáhli nesprávnou odpověď Claude z výše uvedeného a přidali další část konverzace, ve které žádáme Claude, aby opravil svou předchozí odpověď.

In [ ]:
second_user = "Prosím, najděte náhrady za všechna 'slova', která nejsou skutečnými slovy."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Vytisknout odpověď Claude
print("------------------------ Kompletní pole zpráv s náhradou proměnných ------------------------")
print(messages)
print("\n------------------------------------- Odpověď Claude -------------------------------------")
print(get_completion(messages))

Ale přepracovává Claude svou odpověď jen proto, že jsme mu to řekli? Co když začneme s již správnou odpovědí? Ztratí Claude svou jistotu? Zde jsme na místo `first_response` umístili správnou odpověď a požádali ho, aby to znovu zkontroloval.

In [ ]:
```python
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Vytisknout odpověď Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))
```

Můžete si všimnout, že pokud několikrát vygenerujete odpověď z výše uvedeného bloku, Claude většinou nechává slova tak, jak jsou, ale občas je přesto změní, i když jsou všechna již správná. Co můžeme udělat, abychom to zmírnili? Podle kapitoly 8 můžeme dát Claudovi možnost úniku! Zkusme to ještě jednou.

In [ ]:
```python
first_user = "Vyjmenuj deset slov, která všechna končí přesně písmeny 'ab'."

first_response = """Zde je 10 slov, která končí písmeny 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Prosím, najdi náhrady za všechna 'slova', která nejsou skutečnými slovy. Pokud jsou všechna slova skutečná, vrať původní seznam."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Vytiskni odpověď Claude
print("------------------------ Kompletní pole zpráv s náhradami proměnných ------------------------")
print(messages)
print("\n------------------------------------- Odpověď Claude -------------------------------------")
print(get_completion(messages))
```

Zkuste několikrát vygenerovat odpovědi z výše uvedeného kódu, abyste viděli, že Claude je nyní mnohem lepší v dodržování svých zásad.

Můžete také použít prompt chaining k **požádání Clauda, aby své odpovědi vylepšil**. Níže jsme požádali Clauda, aby nejprve napsal příběh, a pak vylepšil příběh, který napsal. Vaše osobní preference se mohou lišit, ale mnozí by mohli souhlasit, že druhá verze od Clauda je lepší.

Nejprve vygenerujeme první verzi příběhu od Clauda.

In [ ]:
```python
# Počáteční prompt
first_user = "Write a three-sentence short story about a girl who likes to run."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Uložení a vytištění Claudeovy odpovědi
first_response = get_completion(messages)
print(first_response)
```

Nyní nechme Claude vylepšit jeho první návrh.

In [ ]:
second_user = "Make the story better."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Vytisknout odpověď Claude
print("------------------------ Kompletní pole zpráv s náhradou proměnných ------------------------")
print(messages)
print("\n------------------------------------- Odpověď Claude -------------------------------------")
print(get_completion(messages))

Tato forma substituce je velmi silná. Používáme substituční zástupce k předávání seznamů, slov, předchozích odpovědí Claude a podobně. Můžete také **použít substituci k tomu, co nazýváme "volání funkcí", což znamená požádat Claude, aby provedl nějakou funkci, a poté vzít výsledky této funkce a požádat Claude, aby s výsledky udělal ještě více**. Funguje to jako jakákoli jiná substituce. Více o tom v dalším dodatku.

Níže je ještě jeden příklad, jak vzít výsledky jednoho volání Claude a zapojit je do dalšího, delšího volání. Začněme s prvním promptem (který tentokrát zahrnuje předvyplnění odpovědi Claude).

In [ ]:
```python
first_user = """Najdi všechna jména z následujícího textu:

"Hey, Jesse. To jsem já, Erin. Volám ohledně večírku, který Joey pořádá zítra. Keisha řekla, že přijde, a myslím, že Mel tam bude také."""

prefill = "<names>"

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill
    
    }
]

# Uložení a tisk odpovědi od Claude
first_response = get_completion(messages)
print("------------------------ Kompletní pole zpráv s náhradou proměnných ------------------------")
print(messages)
print("\n------------------------------------- Odpověď od Claude -------------------------------------")
print(first_response)
```

Pojďme tento seznam jmen předat do dalšího promptu.

In [ ]:
second_user = "Alphabetize the list."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill + "\n" + first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Vytisknout odpověď Claude
print("------------------------ Kompletní pole zpráv s náhradou proměnných ------------------------")
print(messages)
print("\n------------------------------------- Odpověď Claude -------------------------------------")
print(get_completion(messages))

Nyní, když jste se naučili o prompt chaining, přejděte do Přílohy 10.2, abyste se naučili, jak implementovat volání funkcí pomocí prompt chaining.

---

## Příklad hřiště

Toto je prostor, kde můžete volně experimentovat s příklady promptů uvedenými v této lekci a upravovat prompty, abyste viděli, jak to může ovlivnit odpovědi Claude.

In [ ]:
```python
# Počáteční prompt
first_user = "Name ten words that all end with the exact letters 'ab'."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Uložení a vytištění odpovědi od Claude
first_response = get_completion(messages)
print(first_response)
```

In [ ]:
second_user = "Please find replacements for all 'words' that are not real words."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Vytisknout odpověď Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

In [ ]:
```python
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Vytisknout odpověď Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))
```

In [ ]:
```python
first_user = "Vyjmenuj deset slov, která všechna končí přesně písmeny 'ab'."

first_response = """Zde je 10 slov, která končí písmeny 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Najděte prosím náhrady za všechna 'slova', která nejsou skutečnými slovy. Pokud jsou všechna slova skutečná, vraťte původní seznam."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Vytisknout odpověď Claude
print("------------------------ Kompletní pole zpráv s náhradami proměnných ------------------------")
print(messages)
print("\n------------------------------------- Odpověď Claude -------------------------------------")
print(get_completion(messages))
```

In [ ]:
```python
# Počáteční prompt
first_user = "Write a three-sentence short story about a girl who likes to run."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Uložení a vytištění odpovědi od Claude
first_response = get_completion(messages)
print(first_response)
```

In [ ]:
second_user = "Make the story better."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Vytisknout odpověď Claude
print("------------------------ Pole zpráv s proměnnými náhradami ------------------------")
print(messages)
print("\n------------------------------------- Odpověď Claude -------------------------------------")
print(get_completion(messages))

In [ ]:
```python
first_user = """Najdi všechna jména z následujícího textu:

"Hey, Jesse. To jsem já, Erin. Volám ohledně večírku, který Joey pořádá zítra. Keisha řekla, že přijde, a myslím, že Mel tam bude také."""

prefill = "<names>"

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill
    
    }
]

# Uložení a tisk odpovědi od Claude
first_response = get_completion(messages)
print("------------------------ Kompletní pole zpráv s náhradou proměnných ------------------------")
print(messages)
print("\n------------------------------------- Odpověď od Claude -------------------------------------")
print(first_response)
```

In [ ]:
second_user = "Alphabetize the list."

# Pole zpráv API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill + "\n" + first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Vytisknout odpověď od Claude
print("------------------------ Kompletní pole zpráv s náhradou proměnných ------------------------")
print(messages)
print("\n------------------------------------- Odpověď od Claude -------------------------------------")
print(get_completion(messages))